# Trip planner agent

In [12]:
%pip install -q langchain langchain-classic langchain-groq ddgs tavily-python python-dotenv requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import os
from datetime import date

import requests
from ddgs import DDGS
from dotenv import find_dotenv, load_dotenv
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from tavily import TavilyClient

load_dotenv(find_dotenv(usecwd=True))              # on your computer: reads the .env file

try:                                               # on Google Colab: reads the keys from Secrets
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
except Exception:
    pass

tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

In [14]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful trip-planning assistant. Today's date is " + str(date.today()) + ". "
            "For the weather in a city: first call get_coordinates, then call get_weather with the "
            "latitude and longitude it returns. Never guess coordinates. "
            "Use convert_currency for every money conversion. "
            "Use calculator for every sum, difference or total of money: never add numbers yourself. "
            "Use search_web to find pages, then read the best one with extract_page, using a link "
            "from the search results. Never invent a link. "
            "If a tool returns an Error, tell the user what failed. "
            "Answer in four to six short sentences and end with the link of the page you used.",
        ),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)
model = ChatGroq(model="openai/gpt-oss-20b", temperature=0.3, reasoning_effort="low")

In [15]:
@tool
def search_web(query: str) -> str:
    """Search the web and return the top results with a short summary and a link for each."""
    try:
        results = DDGS().text(query, max_results=3)
        return "\n".join(r["title"] + ": " + r["body"][:300] + " (" + r["href"] + ")" for r in results)
    except Exception:
        return "Error: the search failed. Try different words."

In [16]:
@tool
def extract_page(url: str, topic: str) -> str:
    """Open a web page from its link and return the text about the topic. Use a link from search_web."""
    try:
        pages = tavily.extract(urls=[url], query=topic, format="text")["results"]
        return "Page: " + url + "\n" + pages[0]["raw_content"][:2500]
    except Exception:
        return "Error: that page could not be read. Use a full link taken from a search result."

In [17]:
@tool
def get_coordinates(city: str) -> str:
    """Find the latitude and longitude of a city, for example 'Pokhara'."""
    try:
        place = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1},
            timeout=10,
        ).json()["results"][0]
        return place["name"] + ": latitude " + str(round(place["latitude"], 2)) + ", longitude " + str(round(place["longitude"], 2))
    except Exception:
        return "Error: could not find the city " + city + "."

In [18]:
@tool
def get_weather(latitude: float, longitude: float) -> str:
    """Get today's weather at a latitude and longitude. Get the numbers from get_coordinates first."""
    try:
        weather = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": latitude,
                "longitude": longitude,
                "current": "temperature_2m,wind_speed_10m",
                "daily": "precipitation_probability_max",
                "forecast_days": 1,
            },
            timeout=10,
        ).json()
        return (
            str(weather["current"]["temperature_2m"]) + " C, wind " + str(weather["current"]["wind_speed_10m"])
            + " km/h, chance of rain today " + str(weather["daily"]["precipitation_probability_max"][0]) + " percent."
        )
    except Exception:
        return "Error: could not get the weather. Check the latitude and longitude."

In [19]:
@tool
def convert_currency(amount: float, from_code: str, to_code: str) -> str:
    """Convert an amount of money from one currency to another, for example 100 USD to NPR."""
    try:
        rate = requests.get("https://open.er-api.com/v6/latest/" + from_code.upper(), timeout=10).json()["rates"][to_code.upper()]
        return str(amount) + " " + from_code.upper() + " = " + str(round(amount * rate, 2)) + " " + to_code.upper()
    except Exception:
        return "Error: could not convert. Use currency codes like USD, NPR or EUR."

In [20]:
@tool
def calculator(expression: str) -> str:
    """Calculate a plain arithmetic expression like '50000 - 3 * 4000 - 3 * 1500'. Use it for every expense total."""
    if not set(expression.replace(",", "")) <= set("0123456789+-*/(). "):
        return "Error: use only numbers and + - * / ( )."
    try:
        return str(round(eval(expression.replace(",", ""), {"__builtins__": {}}), 2))
    except Exception:
        return "Error: could not calculate that expression."

In [21]:
tools = [search_web, extract_page, get_coordinates, get_weather, convert_currency, calculator]

agent = create_tool_calling_agent(model, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=12)

In [24]:
question = ("I have 50,000 NPR for 3 days in Pokhara. The hotel is 4,000 NPR a night and food is 1,500 NPR a day. "
            "How much is left after 3 days, in NPR and in US dollars? Also, what is the weather, and what is one thing to do?")
result = agent_executor.invoke({"input": question})
print("\nFinal answer:\n" + result["output"])



> Entering new AgentExecutor chain...

Invoking: `search_web` with `{'query': 'Pokhara weather today'}`


Pokhara, Gandaki, Nepal Hourly Weather | AccuWeather: Hourly weather forecast in Pokhara, Gandaki, Nepal. Check current conditions in Pokhara, Gandaki, Nepal with radar, hourly, and more. (https://www.accuweather.com/en/np/pokhara/243170/hourly-weather-forecast/243170)
Pokhara Weather: Today's and tonight's Pokhara, Gandaki Province, Nepal weather forecast, weather conditions and Doppler radar from The Weather Channel and Weather.com (https://weather.com/np/gandaki-province/city/pokhara/today)
Weather Today for Pokhara, Gandaki, Nepal | AccuWeather: Everything you need to know about today's weather in Pokhara, Gandaki, Nepal. High/Low, Precipitation Chances, Sunrise/Sunset, and today's Temperature History. (https://www.accuweather.com/en/np/pokhara/243170/weather-today/243170)
Invoking: `get_coordinates` with `{'city': 'Pokhara'}`


Pokhara: latitude 28.27, longitude 83.97
Invoki

In [26]:
# Using with chat history
agent_executor.invoke(
    {
        "input": "i will stay at hotel and go for a boating and my budget is 100000 nrp and i want to know the weather and how much is left after 4 days staying in pokhara calculate the total expense based on the current hotel and food prices of pokhara also the boating price",
        "chat_history": [
            HumanMessage(content=question),
            AIMessage(content=result["output"]),
        ],
    }
)



> Entering new AgentExecutor chain...

Invoking: `search_web` with `{'query': 'Pokhara hotel price per night 2026'}`


Pokhara Dry Season Hotel Price: 2026 Peak Season Cost Guide: Planning a trip to Nepal? Learn about the Pokhara dry season hotel price and peak season costs for 2026. Get budget tips, neighborhood guides, and price tables. (https://travelstar360.com/pokhara-dry-season-hotel-price/)
Pokhara Lakeside Hotel Price: 2026 Guide to Stay Costs: Discover the latest Pokhara lakeside hotel price for 2026. Compare budget, mid-range, and luxury lakeside Pokhara stay cost options for your Nepal trip. (https://travelstar360.com/pokhara-lakeside-hotel-price-guide/)
Best Hotels in Pokhara 2026/27: Lakeside Resorts Guide: All 17 Pokhara partner hotels ranked by area and price for 2026/27: Lakeside resorts, Sarangkot sunrise lodges and hilltop retreats, with real nightly rate (https://enticinghimalayas.com/en/blog/best-hotels-in-pokhara-202627-lakeside-resorts-guide)
Invoking: `extract_

{'input': 'i will stay at hotel and go for a boating and my budget is 100000 nrp and i want to know the weather and how much is left after 4 days staying in pokhara calculate the total expense based on the current hotel and food prices of pokhara also the boating price',
 'chat_history': [HumanMessage(content='I have 50,000 NPR for 3 days in Pokhara. The hotel is 4,000 NPR a night and food is 1,500 NPR a day. How much is left after 3 days, in NPR and in US dollars? Also, what is the weather, and what is one thing to do?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='You start with 50,000\u202fNPR.  \nThe hotel costs 4,000\u202fNPR per night, so 12,000\u202fNPR for three nights.  \nFood costs 1,500\u202fNPR per day, so 4,500\u202fNPR for three days.  \nAfter subtracting 12,000\u202fNPR + 4,500\u202fNPR from 50,000\u202fNPR, you have 33,500\u202fNPR left.  \nThat amount equals about 217.25\u202fUSD.  \nToday’s weather in Pokhara is 17.6\u202f°C with a 25\u202f% chanc